In [ ]:
"""
Add real human-annotated Xu et al. (2015) Clinical Citation Sentiment
minority-class examples to final_citation_non_citation_dataset.csv.

Selection policy:
- keep only NEGATIVE and POSITIVE;
- keep only rows with exactly one THISCITATION and zero OTHERCITATION;
- remove corpus-specific citation placeholder token from text;
- exclude duplicates and exact overlaps with the existing dataset;
- preserve the six-column final schema.

IMPORTANT FOR FAIR MODEL COMPARISON:
For an experiment directly comparable with a previous Athar-based validation/test
result, append these external rows ONLY to the training split. Do not randomly
redistribute them into validation/test.
"""

In [ ]:
import os
# Set WANDB_API_KEY as an environment variable (never commit an API key).

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pickle
import re
import unicodedata
import pandas as pd
import numpy as np

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER")

FINAL_IN = (
    BASE_DIR
    / "athar_acl_ocl_outputs"
    / "_final_citation_non_citation_dataset.csv"
)

CLINICAL_DIR = (
    BASE_DIR
    / "clinical_citation_dataset_relevant_files"
)

CLINICAL_TEXT = CLINICAL_DIR / "285.txt"
CLINICAL_LABELS = CLINICAL_DIR / "labels.p"

OUTPUT_DIR = (
    BASE_DIR
    / "athar_acl_ocl_outputs_with_clinical"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_MERGED = OUTPUT_DIR / "_final_citation_non_citation_plus_clinical_real.csv"
OUT_CLINICAL = OUTPUT_DIR / "clinical_xu2015_real_minority_citations.csv"
OUT_AUDIT = OUTPUT_DIR / "clinical_xu2015_extraction_audit.csv"

FINAL_COLUMNS = [
    "Source_Paper_ID",
    "sentence",
    "is_citation",
    "polarity",
    "source",
    "sampling_status",
]

In [ ]:
def normalize_for_overlap(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def clean_clinical_sentence(text):
    text = str(text).replace("THISCITATION", " [TARGET_CITATION] ").replace("OTHERCITATION", " [OTHER_CITATION] ")
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)
    return text.strip()

In [ ]:
existing = pd.read_csv(FINAL_IN)
assert list(existing.columns) == FINAL_COLUMNS

with open(CLINICAL_TEXT, "r", encoding="utf-8", errors="replace") as f:
    clinical_lines = [line.rstrip("\r\n") for line in f]
clinical_lines = [x for x in clinical_lines if x != ""]

In [ ]:
# Only load labels.p if it came from the trusted official corpus archive.
with open(CLINICAL_LABELS, "rb") as f:
    labels_array = pickle.load(f, encoding="latin1")

labels = [
    x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x)
    for x in labels_array
]

assert len(clinical_lines) == 4182
assert len(labels) == 4182

clinical = pd.DataFrame({
    "clinical_row_index": np.arange(1, len(clinical_lines) + 1),
    "sentence_raw": clinical_lines,
    "label_raw": labels,
})

clinical["n_thiscitation"] = clinical["sentence_raw"].str.count("THISCITATION")
clinical["n_othercitation"] = clinical["sentence_raw"].str.count("OTHERCITATION")

selected = clinical[
    (clinical["n_thiscitation"] == 1)
    & (clinical["label_raw"].isin(["NEGATIVE", "POSITIVE"]))
].copy()

selected["sentence"] = selected["sentence_raw"].map(clean_clinical_sentence)
selected["polarity"] = selected["label_raw"].map({
    "NEGATIVE": 0,
    "POSITIVE": 2,
}).astype("int8")

selected["_norm"] = selected["sentence"].map(normalize_for_overlap)

#conflicts = selected.groupby("_norm")["polarity"].nunique()
#assert not (conflicts > 1).any(), "Conflicting sentence-level labels found."

# Εντοπισμός sentences που μετά την αντικατάσταση των citation markers
# έχουν περισσότερα από ένα διαφορετικά polarity labels.
label_counts = (
    selected
    .groupby("_norm")["polarity"]
    .nunique()
)

conflicting_norms = set(
    label_counts[label_counts > 1].index
)

print(
    "Conflicting normalized sentences που θα αφαιρεθούν:",
    len(conflicting_norms)
)

selected = selected[
    ~selected["_norm"].isin(conflicting_norms)
].copy()


selected = (
    selected
    .sort_values("clinical_row_index")
    .drop_duplicates(subset=["_norm"], keep="first")
    .copy()
)

existing_norm = set(existing["sentence"].map(normalize_for_overlap))
selected["overlap_with_existing"] = selected["_norm"].isin(existing_norm)

audit = selected.copy()

selected = selected[
    ~selected["overlap_with_existing"]
].copy()

clinical_final = pd.DataFrame({
    "Source_Paper_ID": "CLINICAL_XU2015_SOURCE_ID_UNAVAILABLE",
    "sentence": selected["sentence"],
    "is_citation": np.int8(1),
    "polarity": selected["polarity"].astype("int8"),
    "source": "Clinical_Citation_Sentiment_Xu2015",
    "sampling_status": "external_real_human_annotated",
})

merged = pd.concat(
    [existing[FINAL_COLUMNS], clinical_final[FINAL_COLUMNS]],
    ignore_index=True,
)

merged["is_citation"] = merged["is_citation"].astype("int8")
merged["polarity"] = merged["polarity"].astype("int8")

merged = merged.sample(frac=1.0, random_state=42).reset_index(drop=True)

assert list(merged.columns) == FINAL_COLUMNS
assert not merged[FINAL_COLUMNS].isna().any().any()
assert set(clinical_final["polarity"].unique()).issubset({0, 2})
assert (clinical_final["is_citation"] == 1).all()

clinical_final.to_csv(OUT_CLINICAL, index=False, encoding="utf-8-sig")
audit.to_csv(OUT_AUDIT, index=False, encoding="utf-8-sig")
merged.to_csv(OUT_MERGED, index=False, encoding="utf-8-sig")

print("Original rows:", len(existing))
print("Clinical rows added:", len(clinical_final))
print("\nClinical added:")
print(clinical_final["polarity"].value_counts().sort_index())
print("\nFinal distribution:")
print(merged["polarity"].value_counts().sort_index())
print("\nFinal rows:", len(merged))
print("\nSaved:", OUT_MERGED)